In [1]:
import os
import ast
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from xgboost import XGBClassifier


pd.set_option("display.max_columns", None)

In [33]:
# data path
DATA_PATH = r"D:\lol draft analyzer - datascientest\AA- toutes les donnees au propre - lecture-ecriture"

# Games
GAMES_PATH = f"{DATA_PATH}\\les matchs\\200k_games\\draft_simple.csv"
df_games = pd.read_csv(GAMES_PATH)

# Champions
CHAMPS_PATH = f"{DATA_PATH}\\les stats champions\\champions_15.1.1_15.24.1.csv"
df_champs = pd.read_csv(CHAMPS_PATH, encoding="utf-8")

# WR simples
SIMPLE_WR_PATH = f"{DATA_PATH}\\les winrates simples\\données raffinées\\df_Simple_WR_FULL.csv"
df_simple_wr = pd.read_csv(SIMPLE_WR_PATH, encoding="utf-8")

# WR complets
# COMPLET_WR_PATH = f"{DATA_PATH}\\les winrates matchups\\WR_complet.csv"
# df_complet_wr = pd.read_csv(COMPLET_WR_PATH, encoding="utf-8")


C:\Users\samue\AppData\Local\Temp\ipykernel_33192\3050380018.py:6: DtypeWarning: Columns (10,11,12,13,14,20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df_games = pd.read_csv(GAMES_PATH)


In [ ]:
# différents scalers
# min/max, standard, (robust, quantile, power)

# différents encoders
# one-hot, target, ordinal

In [13]:
df_games.head(10)

,match_id,serveur,patch,elo,blue_side_win,blue_top_champion,blue_jungle_champion,blue_mid_champion,blue_adc_champion,blue_support_champion,blue_top_puuid,blue_jungle_puuid,blue_mid_puuid,blue_adc_puuid,blue_support_puuid,red_top_champion,red_jungle_champion,red_mid_champion,red_adc_champion,red_support_champion,red_top_puuid,red_jungle_puuid,red_mid_puuid,red_adc_puuid,red_support_puuid,blue_ban_1,blue_ban_2,blue_ban_3,blue_ban_4,blue_ban_5,red_ban_1,red_ban_2,red_ban_3,red_ban_4,red_ban_5
0,EUW1_7517331727,euw1,15.17.708.5788,HIGH_ELO,False,Sett,Talon,Diana,Jinx,Rell,NaN,NaN,NaN,NaN,NaN,Yasuo,FiddleSticks,Cassiopeia,Aphelios,Bard,NaN,NaN,NaN,NaN,NaN,Lulu,Fiora,Rengar,Vex,Poppy,Dr. Mundo,Garen,Milio,Zoe,Draven
1,EUW1_7509322616,euw1,15.17.706.7412,HIGH_ELO,False,Mordekaiser,Qiyana,Yasuo,Corki,Nami,NaN,NaN,NaN,NaN,NaN,Jax,Hecarim,Syndra,Senna,Bard,NaN,NaN,NaN,NaN,NaN,Yunara,Pyke,Gwen,Kayle,Smolder,Master Yi,Pantheon,Twitch,Draven,Fiora
2,EUW1_7509243675,euw1,15.17.706.7412,HIGH_ELO,False,Smolder,Kindred,Galio,Jinx,Milio,NaN,NaN,NaN,NaN,NaN,Quinn,Kayn,Xerath,MissFortune,Rakan,NaN,NaN,NaN,NaN,NaN,Dr. Mundo,Rengar,Fiora,Master Yi,Yasuo,Master Yi,Twitch,Akali,Draven,Aatrox
3,EUW1_7509193063,euw1,15.17.706.7412,HIGH_ELO,False,Warwick,Nidalee,Akali,Yunara,Rakan,NaN,NaN,NaN,NaN,NaN,Chogath,RekSai,Xerath,Kaisa,Sona,NaN,NaN,NaN,NaN,NaN,Sivir,Qiyana,Naafiri,Draven,Twitch,Fiddlesticks,Blitzcrank,Mel,Volibear,Darius
4,EUW1_7508909082,euw1,15.17.706.7412,HIGH_ELO,False,Aurora,XinZhao,Twitch,Yunara,Rakan,NaN,NaN,NaN,NaN,NaN,Jayce,Sett,Akali,Lucian,Milio,NaN,NaN,NaN,NaN,NaN,Draven,Kassadin,Shaco,Ahri,Senna,Nocturne,Evelynn,Galio,Yasuo,Riven
5,EUW1_7508869243,euw1,15.17.706.7412,HIGH_ELO,True,Heimerdinger,Trundle,Yasuo,Xerath,Maokai,NaN,NaN,NaN,NaN,NaN,Aurora,Hecarim,Malzahar,Ashe,Nidalee,NaN,NaN,NaN,NaN,NaN,Blitzcrank,Olaf,Mel,Yunara,Qiyana,Draven,Ziggs,Ivern,Akshan,Qiyana
6,EUW1_7508851515,euw1,15.17.706.7412,HIGH_ELO,False,Skarner,Talon,Fizz,Smolder,Rakan,NaN,NaN,NaN,NaN,NaN,Darius,Nidalee,Katarina,Yunara,Taric,NaN,NaN,NaN,NaN,NaN,Aphelios,Qiyana,Vayne,Galio,NaN,Draven,Karma,Pantheon,Yasuo,Renekton
7,EUW1_7507963628,euw1,15.16.706.7476,HIGH_ELO,False,Urgot,Shaco,Syndra,Corki,Pantheon,NaN,NaN,NaN,NaN,NaN,Aurora,Qiyana,Yasuo,Samira,Sylas,NaN,NaN,NaN,NaN,NaN,Twitch,Fiora,Katarina,Gwen,Poppy,Milio,K'Sante,Lulu,Yunara,LeBlanc
8,EUW1_7506939511,euw1,15.16.706.7476,HIGH_ELO,True,Illaoi,Volibear,Syndra,Ezreal,Neeko,NaN,NaN,NaN,NaN,NaN,Poppy,Gwen,MonkeyKing,Jhin,Xerath,NaN,NaN,NaN,NaN,NaN,Nautilus,Twitch,Viego,Kayle,Ambessa,Pyke,Yasuo,Katarina,Yunara,NaN
9,EUW1_7512192924,euw1,15.17.708.2856,HIGH_ELO,True,LeeSin,Hecarim,Kassadin,Draven,Rell,NaN,NaN,NaN,NaN,NaN,Gangplank,Diana,Jhin,Yunara,Galio,NaN,NaN,NaN,NaN,NaN,Braum,Twitch,Shaco,Talon,Sylas,Rengar,Vladimir,Zed,Qiyana,Mel


In [43]:
df_simple_wr["patch"] = (
    df_simple_wr["patch"]
    .astype(str)
    .str.split(".")
    .str[0]
)

In [ ]:
df_simple_wr["patch"].unique()

array(['16', '15'], dtype=object)

In [38]:
df_games["patch"] = (
    df_games["patch"]
    .str.split(".")
    .str[:1]
    .str.join(".")
)

In [39]:
df_games.head()
df_games["patch"].unique()
# df_games.shape

array(['15'], dtype=object)

In [46]:
df_games.head()

,match_id,serveur,patch,elo,blue_side_win,blue_top_champion,blue_jungle_champion,blue_mid_champion,blue_adc_champion,blue_support_champion,blue_top_puuid,blue_jungle_puuid,blue_mid_puuid,blue_adc_puuid,blue_support_puuid,red_top_champion,red_jungle_champion,red_mid_champion,red_adc_champion,red_support_champion,red_top_puuid,red_jungle_puuid,red_mid_puuid,red_adc_puuid,red_support_puuid,blue_ban_1,blue_ban_2,blue_ban_3,blue_ban_4,blue_ban_5,red_ban_1,red_ban_2,red_ban_3,red_ban_4,red_ban_5
0,EUW1_7517331727,euw1,15,HIGH_ELO,False,Sett,Talon,Diana,Jinx,Rell,NaN,NaN,NaN,NaN,NaN,Yasuo,FiddleSticks,Cassiopeia,Aphelios,Bard,NaN,NaN,NaN,NaN,NaN,Lulu,Fiora,Rengar,Vex,Poppy,Dr. Mundo,Garen,Milio,Zoe,Draven
1,EUW1_7509322616,euw1,15,HIGH_ELO,False,Mordekaiser,Qiyana,Yasuo,Corki,Nami,NaN,NaN,NaN,NaN,NaN,Jax,Hecarim,Syndra,Senna,Bard,NaN,NaN,NaN,NaN,NaN,Yunara,Pyke,Gwen,Kayle,Smolder,Master Yi,Pantheon,Twitch,Draven,Fiora
2,EUW1_7509243675,euw1,15,HIGH_ELO,False,Smolder,Kindred,Galio,Jinx,Milio,NaN,NaN,NaN,NaN,NaN,Quinn,Kayn,Xerath,MissFortune,Rakan,NaN,NaN,NaN,NaN,NaN,Dr. Mundo,Rengar,Fiora,Master Yi,Yasuo,Master Yi,Twitch,Akali,Draven,Aatrox
3,EUW1_7509193063,euw1,15,HIGH_ELO,False,Warwick,Nidalee,Akali,Yunara,Rakan,NaN,NaN,NaN,NaN,NaN,Chogath,RekSai,Xerath,Kaisa,Sona,NaN,NaN,NaN,NaN,NaN,Sivir,Qiyana,Naafiri,Draven,Twitch,Fiddlesticks,Blitzcrank,Mel,Volibear,Darius
4,EUW1_7508909082,euw1,15,HIGH_ELO,False,Aurora,XinZhao,Twitch,Yunara,Rakan,NaN,NaN,NaN,NaN,NaN,Jayce,Sett,Akali,Lucian,Milio,NaN,NaN,NaN,NaN,NaN,Draven,Kassadin,Shaco,Ahri,Senna,Nocturne,Evelynn,Galio,Yasuo,Riven


In [45]:
df_simple_wr.head()

,elo,server,patch,label,name,role,role_pickrate,tier,winrate,winrate_evol,pickrate,games
0,TOUT,TOUT,16,1,Briar,jun,91.1%,S+,52.8%,+1.5%,6.7%,599 576
1,TOUT,TOUT,16,2,Miss Fortune,adc,97.0%,S+,51.7%,-0.0%,15.5%,1 386 160
2,TOUT,TOUT,16,3,Malzahar,mid,92.2%,S+,52.3%,+0.0%,8.1%,724 180
3,TOUT,TOUT,16,4,Swain,adc,18.6%,S+,54.6%,-0.1%,1.7%,154 988
4,TOUT,TOUT,16,5,Jinx,adc,99.5%,S+,51.6%,-0.2%,16.6%,1 485 175


In [47]:
df_simple_wr["server"].unique()

array(['TOUT', nan, 'KR', 'EUW', 'VN'], dtype=object)

In [51]:
df_games["serveur"].unique()

array(['euw1', 'kr', 'eun1'], dtype=object)

In [ ]:
df_simple_wr[df_simple_wr["server"].isna()].shape
#correspond au serveur North america (NA est devenu nan)

(444, 12)

In [53]:
cols_for_match = [
    "elo",
    "serveur",   # ou "server" selon ton df
    "patch",
    
    "red_top_champion",
    "red_jungle_champion",
    "red_mid_champion",
    "red_adc_champion",
    "red_support_champion",
    
    "blue_side_win",
    
    "blue_top_champion",
    "blue_jungle_champion",
    "blue_mid_champion",
    "blue_adc_champion",
    "blue_support_champion",
]

In [54]:
cols_existing = [c for c in cols_for_match if c in df_games.columns]

df_matchs = df_games[cols_existing]

df_matchs.head()

,elo,serveur,patch,red_top_champion,red_jungle_champion,red_mid_champion,red_adc_champion,red_support_champion,blue_side_win,blue_top_champion,blue_jungle_champion,blue_mid_champion,blue_adc_champion,blue_support_champion
0,HIGH_ELO,euw1,15,Yasuo,FiddleSticks,Cassiopeia,Aphelios,Bard,False,Sett,Talon,Diana,Jinx,Rell
1,HIGH_ELO,euw1,15,Jax,Hecarim,Syndra,Senna,Bard,False,Mordekaiser,Qiyana,Yasuo,Corki,Nami
2,HIGH_ELO,euw1,15,Quinn,Kayn,Xerath,MissFortune,Rakan,False,Smolder,Kindred,Galio,Jinx,Milio
3,HIGH_ELO,euw1,15,Chogath,RekSai,Xerath,Kaisa,Sona,False,Warwick,Nidalee,Akali,Yunara,Rakan
4,HIGH_ELO,euw1,15,Jayce,Sett,Akali,Lucian,Milio,False,Aurora,XinZhao,Twitch,Yunara,Rakan


In [55]:
col = "blue_side_win"

df_matchs = df_matchs[[c for c in df_matchs.columns if c != col] + [col]]

In [56]:
df_matchs.head()

,elo,serveur,patch,red_top_champion,red_jungle_champion,red_mid_champion,red_adc_champion,red_support_champion,blue_top_champion,blue_jungle_champion,blue_mid_champion,blue_adc_champion,blue_support_champion,blue_side_win
0,HIGH_ELO,euw1,15,Yasuo,FiddleSticks,Cassiopeia,Aphelios,Bard,Sett,Talon,Diana,Jinx,Rell,False
1,HIGH_ELO,euw1,15,Jax,Hecarim,Syndra,Senna,Bard,Mordekaiser,Qiyana,Yasuo,Corki,Nami,False
2,HIGH_ELO,euw1,15,Quinn,Kayn,Xerath,MissFortune,Rakan,Smolder,Kindred,Galio,Jinx,Milio,False
3,HIGH_ELO,euw1,15,Chogath,RekSai,Xerath,Kaisa,Sona,Warwick,Nidalee,Akali,Yunara,Rakan,False
4,HIGH_ELO,euw1,15,Jayce,Sett,Akali,Lucian,Milio,Aurora,XinZhao,Twitch,Yunara,Rakan,False


In [58]:
df_matchs["elo"].unique()

array(['HIGH_ELO', 'UNKNOWN', 'CHALLENGER', 'GRANDMASTER', 'MASTER',
       'DIAMOND'], dtype=object)

In [57]:
df_simple_wr.head()

,elo,server,patch,label,name,role,role_pickrate,tier,winrate,winrate_evol,pickrate,games
0,TOUT,TOUT,16,1,Briar,jun,91.1%,S+,52.8%,+1.5%,6.7%,599 576
1,TOUT,TOUT,16,2,Miss Fortune,adc,97.0%,S+,51.7%,-0.0%,15.5%,1 386 160
2,TOUT,TOUT,16,3,Malzahar,mid,92.2%,S+,52.3%,+0.0%,8.1%,724 180
3,TOUT,TOUT,16,4,Swain,adc,18.6%,S+,54.6%,-0.1%,1.7%,154 988
4,TOUT,TOUT,16,5,Jinx,adc,99.5%,S+,51.6%,-0.2%,16.6%,1 485 175


In [59]:
df_simple_wr["elo"].unique()

array(['TOUT', 'Bronze', 'Grandmaster', 'Challenger'], dtype=object)

In [62]:
df_matchs["serveur"].unique()

array(['euw1', 'kr', 'eun1'], dtype=object)

In [60]:
df_simple_wr["server"].unique()

array(['TOUT', nan, 'KR', 'EUW', 'VN'], dtype=object)

In [63]:
import pandas as pd

# Mapping elo / serveur
elo_map = {
    "HIGH_ELO": "TOUT",
    "UNKNOWN": "TOUT",
    "CHALLENGER": "Challenger",
    "GRANDMASTER": "Grandmaster",
    "MASTER": "TOUT",
    "DIAMOND": "TOUT",
}

server_map = {
    "euw1": "EUW",
    "kr": "KR",
    "eun1": "TOUT",  # pas présent dans df_simple_wr
}

# Mapping colonnes champion -> rôle
role_map = {
    "red_top_champion": "top",
    "red_jungle_champion": "jun",
    "red_mid_champion": "mid",
    "red_adc_champion": "adc",
    "red_support_champion": "support",
    "blue_top_champion": "top",
    "blue_jungle_champion": "jun",
    "blue_mid_champion": "mid",
    "blue_adc_champion": "adc",
    "blue_support_champion": "support",
}

# Fonction pour récupérer winrate d'un champion
def get_winrate(champion_name, lane, patch, elo, server):
    # Filtrage
    df_tmp = df_simple_wr[
        (df_simple_wr["name"] == champion_name) &
        (df_simple_wr["role"] == lane) &
        (df_simple_wr["patch"] == patch) &
        (df_simple_wr["elo"] == elo) &
        (df_simple_wr["server"] == server)
    ]
    if len(df_tmp) == 0:
        return "-1%"
    else:
        return df_tmp["winrate"].values[0]

# Création du nouveau df
df_matchs_feat_simple_WR = df_matchs[["elo", "serveur", "patch"]].copy()

# Pour chaque colonne champion, remplacer par winrate
for col, lane in role_map.items():
    df_matchs_feat_simple_WR[col] = df_matchs.apply(
        lambda row: get_winrate(
            row[col],
            lane,
            row["patch"],
            elo_map.get(row["elo"], "TOUT"),
            server_map.get(row["serveur"], "TOUT")
        ),
        axis=1
    )

df_matchs_feat_simple_WR.head()

KeyboardInterrupt: 

In [65]:
import pandas as pd

# -------------------------------
# 1️⃣ Préparer df_simple_wr
# -------------------------------

# Garder uniquement les colonnes nécessaires
df_simple_wr_clean = df_simple_wr[["elo", "server", "patch", "name", "role", "winrate"]].copy()

# Standardiser elo / server pour faciliter le merge
elo_map = {
    "HIGH_ELO": "TOUT",
    "UNKNOWN": "TOUT",
    "CHALLENGER": "Challenger",
    "GRANDMASTER": "Grandmaster",
    "MASTER": "TOUT",
    "DIAMOND": "TOUT",
}

server_map = {
    "euw1": "EUW",
    "kr": "KR",
    "eun1": "TOUT",  # pas présent dans df_simple_wr
}

df_simple_wr_clean["elo"] = df_simple_wr_clean["elo"].str.upper()
df_simple_wr_clean["server"] = df_simple_wr_clean["server"].str.upper()

# -------------------------------
# 2️⃣ Transformer df_matchs en format long
# -------------------------------

role_map = {
    "red_top_champion": "top",
    "red_jungle_champion": "jun",
    "red_mid_champion": "mid",
    "red_adc_champion": "adc",
    "red_support_champion": "support",
    "blue_top_champion": "top",
    "blue_jungle_champion": "jun",
    "blue_mid_champion": "mid",
    "blue_adc_champion": "adc",
    "blue_support_champion": "support",
}

# Ajouter un match_id pour pivot plus tard
df_matchs_long = df_matchs.reset_index().rename(columns={"index": "match_id"})

# Melt : transformer colonnes champion en lignes
df_long = df_matchs_long.melt(
    id_vars=["match_id", "elo", "serveur", "patch", "blue_side_win"],
    value_vars=list(role_map.keys()),
    var_name="champion_role_col",
    value_name="name"
)

# Ajouter colonne role correspondante
df_long["role"] = df_long["champion_role_col"].map(role_map)

# Standardiser elo et serveur pour merge
df_long["elo"] = df_long["elo"].map(elo_map).fillna("TOUT")
df_long["server"] = df_long["serveur"].map(server_map).fillna("TOUT")  # pour merge avec df_simple_wr
df_long["patch"] = df_long["patch"].astype(str)

# -------------------------------
# 3️⃣ Merge vectorisé pour récupérer winrate
# -------------------------------

df_long = df_long.merge(
    df_simple_wr_clean,
    left_on=["elo", "server", "patch", "role", "name"],
    right_on=["elo", "server", "patch", "role", "name"],
    how="left"
)

# Remplacer les NaN par -1%
df_long["winrate"] = df_long["winrate"].fillna("-1%")

# -------------------------------
# 4️⃣ Pivot pour revenir au format large
# -------------------------------

df_matchs_feat_simple_WR = df_long.pivot_table(
    index=["match_id", "elo", "serveur", "patch", "blue_side_win"],
    columns="champion_role_col",
    values="winrate",
    aggfunc="first"
).reset_index()

# Remettre les colonnes dans l'ordre souhaité
cols_order = ["elo", "serveur", "patch"] + list(role_map.keys()) + ["blue_side_win"]
df_matchs_feat_simple_WR = df_matchs_feat_simple_WR[cols_order]

df_matchs_feat_simple_WR.head()

champion_role_col,elo,serveur,patch,red_top_champion,red_jungle_champion,red_mid_champion,red_adc_champion,red_support_champion,blue_top_champion,blue_jungle_champion,blue_mid_champion,blue_adc_champion,blue_support_champion,blue_side_win
0,TOUT,euw1,15,48.8%,-1%,50.5%,50.0%,-1%,52.1%,49.3%,51.4%,50.4%,-1%,False
1,TOUT,euw1,15,49.3%,49.2%,50.8%,-1%,-1%,50.9%,45.3%,50.1%,46.8%,-1%,False
2,TOUT,euw1,15,49.9%,50.1%,51.3%,-1%,-1%,-1%,48.0%,49.9%,50.4%,-1%,False
3,TOUT,euw1,15,-1%,-1%,51.3%,-1%,-1%,52.1%,45.4%,49.1%,46.9%,-1%,False
4,TOUT,euw1,15,46.8%,-1%,49.1%,48.2%,-1%,-1%,-1%,-1%,46.9%,-1%,False


In [66]:
# Remplacer -1% par 50% pour toutes les colonnes champion
champ_cols = list(role_map.keys())

df_matchs_feat_simple_WR[champ_cols] = df_matchs_feat_simple_WR[champ_cols].replace("-1%", "50%")

df_matchs_feat_simple_WR.head()

champion_role_col,elo,serveur,patch,red_top_champion,red_jungle_champion,red_mid_champion,red_adc_champion,red_support_champion,blue_top_champion,blue_jungle_champion,blue_mid_champion,blue_adc_champion,blue_support_champion,blue_side_win
0,TOUT,euw1,15,48.8%,50%,50.5%,50.0%,50%,52.1%,49.3%,51.4%,50.4%,50%,False
1,TOUT,euw1,15,49.3%,49.2%,50.8%,50%,50%,50.9%,45.3%,50.1%,46.8%,50%,False
2,TOUT,euw1,15,49.9%,50.1%,51.3%,50%,50%,50%,48.0%,49.9%,50.4%,50%,False
3,TOUT,euw1,15,50%,50%,51.3%,50%,50%,52.1%,45.4%,49.1%,46.9%,50%,False
4,TOUT,euw1,15,46.8%,50%,49.1%,48.2%,50%,50%,50%,50%,46.9%,50%,False


In [76]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# -------------------------------
# 1️⃣ Définir colonnes champions
# -------------------------------
champ_cols = [
    "red_top_champion", "red_jungle_champion", "red_mid_champion", 
    "red_adc_champion", "red_support_champion",
    "blue_top_champion", "blue_jungle_champion", "blue_mid_champion", 
    "blue_adc_champion", "blue_support_champion"
]

# -------------------------------
# 2️⃣ Supprimer la colonne champion_role_col si présente
# -------------------------------
if "champion_role_col" in df_matchs_feat_simple_WR.columns:
    df_matchs_feat_simple_WR = df_matchs_feat_simple_WR.drop(columns=["champion_role_col"])

# -------------------------------
# 3️⃣ Convertir % en float 0–1
# -------------------------------
df_ml = df_matchs_feat_simple_WR.copy()

for col in champ_cols:
    df_ml[col] = df_ml[col].str.rstrip("%").astype(float) / 100.0

# -------------------------------
# 4️⃣ Créer version normalisée avec MinMaxScaler
# -------------------------------
scaler_minmax = MinMaxScaler()
df_minmax = df_ml.copy()
df_minmax[champ_cols] = scaler_minmax.fit_transform(df_minmax[champ_cols])

# -------------------------------
# 5️⃣ Créer version StandardScaler
# -------------------------------
scaler_standard = StandardScaler()
df_standard = df_ml.copy()
df_standard[champ_cols] = scaler_standard.fit_transform(df_standard[champ_cols])

# -------------------------------
# 6️⃣ Créer version RobustScaler
# -------------------------------
scaler_robust = RobustScaler()
df_robust = df_ml.copy()
df_robust[champ_cols] = scaler_robust.fit_transform(df_robust[champ_cols])

# -------------------------------
# ✅ Tout est prêt
# -------------------------------
print("DF float pour ML :", df_ml.head())
print("==============================")
print("DF MinMax :", df_minmax.head())
print("==============================")
print("DF Standard :", df_standard.head())
print("==============================")
print("DF Robust :", df_robust.head())

DF float pour ML : champion_role_col   elo serveur patch  red_top_champion  red_jungle_champion  \
0                  TOUT    euw1    15             0.488                0.500   
1                  TOUT    euw1    15             0.493                0.492   
2                  TOUT    euw1    15             0.499                0.501   
3                  TOUT    euw1    15             0.500                0.500   
4                  TOUT    euw1    15             0.468                0.500   

champion_role_col  red_mid_champion  red_adc_champion  red_support_champion  \
0                             0.505             0.500                   0.5   
1                             0.508             0.500                   0.5   
2                             0.513             0.500                   0.5   
3                             0.513             0.500                   0.5   
4                             0.491             0.482                   0.5   

champion_role_col  blue_t

In [73]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

# Colonnes à encoder
cat_label_cols = ["serveur", "patch"]
elo_order = ["TOUT", "Bronze", "Silver", "Gold", "Platinum", "Diamond", "Master", "Grandmaster", "Challenger", "HIGH_ELO", "UNKNOWN"]

# Les 4 DF
dfs = {
    "df_ml": df_ml,
    "df_minmax": df_minmax,
    "df_standard": df_standard,
    "df_robust": df_robust
}

# --- 1️⃣ Ordinal encoding pour elo ---
ordinal_encoder = OrdinalEncoder(categories=[elo_order], dtype=int)

for name, df in dfs.items():
    df[["elo"]] = ordinal_encoder.fit_transform(df[["elo"]])

# --- 2️⃣ Label encoding pour serveur et patch ---
label_encoders = {col: LabelEncoder() for col in cat_label_cols}

for name, df in dfs.items():
    for col in cat_label_cols:
        df[col] = label_encoders[col].fit_transform(df[col])

# --- Vérification rapide ---
for name, df in dfs.items():
    print(f"\n{name} head:")
    print(df.head())
    print("==============================")


df_ml head:
champion_role_col  elo  serveur  patch  red_top_champion  red_jungle_champion  \
0                    0        1      0             0.488                0.500   
1                    0        1      0             0.493                0.492   
2                    0        1      0             0.499                0.501   
3                    0        1      0             0.500                0.500   
4                    0        1      0             0.468                0.500   

champion_role_col  red_mid_champion  red_adc_champion  red_support_champion  \
0                             0.505             0.500                   0.5   
1                             0.508             0.500                   0.5   
2                             0.513             0.500                   0.5   
3                             0.513             0.500                   0.5   
4                             0.491             0.482                   0.5   

champion_role_col  blue_t

In [78]:
# Forcer toutes les colonnes à numeric (float ou int)
for col in feature_cols:
    df_ml[col] = pd.to_numeric(df_ml[col], errors='coerce')  # convertit object -> float
    df_minmax[col] = pd.to_numeric(df_minmax[col], errors='coerce')
    df_standard[col] = pd.to_numeric(df_standard[col], errors='coerce')
    df_robust[col] = pd.to_numeric(df_robust[col], errors='coerce')

# Vérification
print(df_ml.dtypes)

champion_role_col
elo                      float64
serveur                  float64
patch                      int64
red_top_champion         float64
red_jungle_champion      float64
red_mid_champion         float64
red_adc_champion         float64
red_support_champion     float64
blue_top_champion        float64
blue_jungle_champion     float64
blue_mid_champion        float64
blue_adc_champion        float64
blue_support_champion    float64
blue_side_win               bool
dtype: object


In [79]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score
from xgboost import XGBClassifier

# Liste des DF
dfs = {
    "df_ml": df_ml,
    "df_minmax": df_minmax,
    "df_standard": df_standard,
    "df_robust": df_robust
}

# Colonnes à utiliser comme features
feature_cols = [c for c in df_ml.columns if c != "blue_side_win"]

# Stocker les résultats
results = []

for name, df in dfs.items():
    print(f"\n--- Training XGBoost on {name} ---")
    
    # Séparer features et target
    X = df[feature_cols]
    y = df["blue_side_win"].astype(int)  # bool -> int pour XGBoost
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Modèle XGBoost
    model = XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    # Prédictions
    y_pred = model.predict(X_test)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"Accuracy: {acc:.4f}, Recall: {rec:.4f}, F1-score: {f1:.4f}")
    
    results.append({
        "DF": name,
        "Accuracy": acc,
        "Recall": rec,
        "F1-score": f1
    })

# Résumé des résultats
results_df = pd.DataFrame(results)
print("\n--- Résumé ---")
print(results_df)


--- Training XGBoost on df_ml ---


c:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [00:06:31] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Accuracy: 0.5218, Recall: 0.8031, F1-score: 0.6343

--- Training XGBoost on df_minmax ---


c:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [00:06:33] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Accuracy: 0.5218, Recall: 0.8031, F1-score: 0.6343

--- Training XGBoost on df_standard ---


c:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [00:06:34] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Accuracy: 0.5218, Recall: 0.8031, F1-score: 0.6343

--- Training XGBoost on df_robust ---


c:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [00:06:35] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Accuracy: 0.5218, Recall: 0.8031, F1-score: 0.6343

--- Résumé ---
            DF  Accuracy    Recall  F1-score
0        df_ml  0.521797  0.803087  0.634349
1    df_minmax  0.521797  0.803087  0.634349
2  df_standard  0.521797  0.803087  0.634349
3    df_robust  0.521797  0.803087  0.634349


In [80]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score
from xgboost import XGBClassifier

# Liste des DataFrames
dfs = {
    "df_ml": df_ml,
    "df_minmax": df_minmax,
    "df_standard": df_standard,
    "df_robust": df_robust
}

# Colonnes features
feature_cols = [c for c in df_ml.columns if c != "blue_side_win"]

# Cutoffs à tester (tu peux ajuster le nombre ou les valeurs)
cutoffs = np.linspace(0.05, 0.25, 10)  # 10 cutoffs de 0.05 à 0.25

for name, df in dfs.items():
    print(f"\n=== Training XGBoost on {name} ===")
    
    # Séparer features / target
    X = df[feature_cols]
    y = df["blue_side_win"].astype(int)
    
    # Train/Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Modèle XGBoost
    model = XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    # --- Predictions ---
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]  # proba de classe 1
    
    # Metrics sur tout le test set
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print(f"\nOverall Test Metrics: Accuracy={acc:.4f}, Recall={rec:.4f}, F1={f1:.4f}")
    
    # --- Metrics pour chaque cutoff ---
    for cutoff in cutoffs:
        lower = 0.5 - cutoff
        upper = 0.5 + cutoff
        
        # On garde les indices hors de l'intervalle [0.5-cutoff, 0.5+cutoff]
        idx_keep = (y_prob < lower) | (y_prob > upper)
        idx_ignore = ~idx_keep
        
        y_test_keep = y_test[idx_keep]
        y_pred_keep = (y_prob[idx_keep] > 0.5).astype(int)  # seuil 0.5 pour la décision
        
        n_eval = len(y_test_keep)
        n_ignore = len(y_test) - n_eval
        
        if n_eval == 0:
            print(f"Cutoff={cutoff:.3f}: No predictions outside interval, skipping metrics")
            continue
        
        acc_cut = accuracy_score(y_test_keep, y_pred_keep)
        rec_cut = recall_score(y_test_keep, y_pred_keep)
        f1_cut = f1_score(y_test_keep, y_pred_keep)
        
        print(f"Cutoff={cutoff:.3f} | Evaluated={n_eval}, Ignored={n_ignore} | "
              f"Accuracy={acc_cut:.4f}, Recall={rec_cut:.4f}, F1={f1_cut:.4f}")


=== Training XGBoost on df_ml ===


c:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [00:12:39] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Overall Test Metrics: Accuracy=0.5218, Recall=0.8031, F1=0.6343
Cutoff=0.050 | Evaluated=5516, Ignored=50684 | Accuracy=0.5489, Recall=0.8440, F1=0.6701
Cutoff=0.072 | Evaluated=1825, Ignored=54375 | Accuracy=0.5699, Recall=0.8277, F1=0.6841
Cutoff=0.094 | Evaluated=717, Ignored=55483 | Accuracy=0.5872, Recall=0.8234, F1=0.6998
Cutoff=0.117 | Evaluated=290, Ignored=55910 | Accuracy=0.5897, Recall=0.8204, F1=0.6972
Cutoff=0.139 | Evaluated=121, Ignored=56079 | Accuracy=0.5868, Recall=0.8429, F1=0.7024
Cutoff=0.161 | Evaluated=45, Ignored=56155 | Accuracy=0.6000, Recall=0.8846, F1=0.7188
Cutoff=0.183 | Evaluated=14, Ignored=56186 | Accuracy=0.5714, Recall=0.8571, F1=0.6667
Cutoff=0.206 | Evaluated=3, Ignored=56197 | Accuracy=0.3333, Recall=0.0000, F1=0.0000
Cutoff=0.228 | Evaluated=3, Ignored=56197 | Accuracy=0.3333, Recall=0.0000, F1=0.0000
Cutoff=0.250 | Evaluated=2, Ignored=56198 | Accuracy=0.0000, Recall=0.0000, F1=0.0000

=== Training XGBoost on df_minmax ===


c:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [00:12:45] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Overall Test Metrics: Accuracy=0.5218, Recall=0.8031, F1=0.6343
Cutoff=0.050 | Evaluated=5516, Ignored=50684 | Accuracy=0.5489, Recall=0.8440, F1=0.6701
Cutoff=0.072 | Evaluated=1825, Ignored=54375 | Accuracy=0.5699, Recall=0.8277, F1=0.6841
Cutoff=0.094 | Evaluated=717, Ignored=55483 | Accuracy=0.5872, Recall=0.8234, F1=0.6998
Cutoff=0.117 | Evaluated=290, Ignored=55910 | Accuracy=0.5897, Recall=0.8204, F1=0.6972
Cutoff=0.139 | Evaluated=121, Ignored=56079 | Accuracy=0.5868, Recall=0.8429, F1=0.7024
Cutoff=0.161 | Evaluated=45, Ignored=56155 | Accuracy=0.6000, Recall=0.8846, F1=0.7188
Cutoff=0.183 | Evaluated=14, Ignored=56186 | Accuracy=0.5714, Recall=0.8571, F1=0.6667
Cutoff=0.206 | Evaluated=3, Ignored=56197 | Accuracy=0.3333, Recall=0.0000, F1=0.0000
Cutoff=0.228 | Evaluated=3, Ignored=56197 | Accuracy=0.3333, Recall=0.0000, F1=0.0000
Cutoff=0.250 | Evaluated=2, Ignored=56198 | Accuracy=0.0000, Recall=0.0000, F1=0.0000

=== Training XGBoost on df_standard ===


c:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [00:12:47] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Overall Test Metrics: Accuracy=0.5218, Recall=0.8031, F1=0.6343
Cutoff=0.050 | Evaluated=5516, Ignored=50684 | Accuracy=0.5489, Recall=0.8440, F1=0.6701
Cutoff=0.072 | Evaluated=1825, Ignored=54375 | Accuracy=0.5699, Recall=0.8277, F1=0.6841
Cutoff=0.094 | Evaluated=717, Ignored=55483 | Accuracy=0.5872, Recall=0.8234, F1=0.6998
Cutoff=0.117 | Evaluated=290, Ignored=55910 | Accuracy=0.5897, Recall=0.8204, F1=0.6972
Cutoff=0.139 | Evaluated=121, Ignored=56079 | Accuracy=0.5868, Recall=0.8429, F1=0.7024
Cutoff=0.161 | Evaluated=45, Ignored=56155 | Accuracy=0.6000, Recall=0.8846, F1=0.7188
Cutoff=0.183 | Evaluated=14, Ignored=56186 | Accuracy=0.5714, Recall=0.8571, F1=0.6667
Cutoff=0.206 | Evaluated=3, Ignored=56197 | Accuracy=0.3333, Recall=0.0000, F1=0.0000
Cutoff=0.228 | Evaluated=3, Ignored=56197 | Accuracy=0.3333, Recall=0.0000, F1=0.0000
Cutoff=0.250 | Evaluated=2, Ignored=56198 | Accuracy=0.0000, Recall=0.0000, F1=0.0000

=== Training XGBoost on df_robust ===


c:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [00:12:49] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Overall Test Metrics: Accuracy=0.5218, Recall=0.8031, F1=0.6343
Cutoff=0.050 | Evaluated=5516, Ignored=50684 | Accuracy=0.5489, Recall=0.8440, F1=0.6701
Cutoff=0.072 | Evaluated=1825, Ignored=54375 | Accuracy=0.5699, Recall=0.8277, F1=0.6841
Cutoff=0.094 | Evaluated=717, Ignored=55483 | Accuracy=0.5872, Recall=0.8234, F1=0.6998
Cutoff=0.117 | Evaluated=290, Ignored=55910 | Accuracy=0.5897, Recall=0.8204, F1=0.6972
Cutoff=0.139 | Evaluated=121, Ignored=56079 | Accuracy=0.5868, Recall=0.8429, F1=0.7024
Cutoff=0.161 | Evaluated=45, Ignored=56155 | Accuracy=0.6000, Recall=0.8846, F1=0.7188
Cutoff=0.183 | Evaluated=14, Ignored=56186 | Accuracy=0.5714, Recall=0.8571, F1=0.6667
Cutoff=0.206 | Evaluated=3, Ignored=56197 | Accuracy=0.3333, Recall=0.0000, F1=0.0000
Cutoff=0.228 | Evaluated=3, Ignored=56197 | Accuracy=0.3333, Recall=0.0000, F1=0.0000
Cutoff=0.250 | Evaluated=2, Ignored=56198 | Accuracy=0.0000, Recall=0.0000, F1=0.0000
